In [1]:
import duckdb

In [2]:
import os
print(os.getcwd())

assert os.path.exists('dados_duckdb.db'), "Banco não encontrado!"
conn = duckdb.connect(database='dados_duckdb.db', read_only=False)

/home/gabriel/Documents/GitHub/PythonDataEngineering_project_pos/scripts


In [5]:
df_ver = conn.execute("SELECT * FROM bronze_produtos").fetchdf()
df_ver.head(10)

,ATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-11-25 19:56:00.952717
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-11-25 19:56:00.952717
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-11-25 19:56:00.952717
3,10003,PREGO,BT10,100,60,z0019_2.csv,2026-11-25 19:56:27.889861
4,10004,SERRA,BT50,100,200,z0019_2.csv,2026-11-25 19:56:27.889861
5,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-11-25 19:56:27.889861


In [6]:
df = conn.execute("""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY MAKTX order by data_ingestao DESC) rank
        FROM bronze_produtos
    ) WHERE rank = 1
    """).fetchdf()
df.head(10)

,ATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,rank
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-11-25 19:56:00.952717,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-11-25 19:56:00.952717,1
2,10003,PREGO,BT10,100,60,z0019_2.csv,2026-11-25 19:56:27.889861,1
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-11-25 19:56:27.889861,1
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-11-25 19:56:27.889861,1


In [7]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'rank'])
df_final = df_final.rename(columns={"ATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"descricao_produto"})
df_final = df_final.rename(columns={"WERKS":"categoria"})
df_final = df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"valor"})
df_final.head(10)

,id,descricao_produto,categoria,id_fornecedor,valor
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10003,PREGO,BT10,100,60
3,10004,SERRA,BT50,100,200
4,10005,MACHADO,BT50,100,100


In [9]:
df_final.dtypes

id                   str
descricao_produto    str
categoria            str
id_fornecedor        str
valor                str
dtype: object

In [11]:
df_format = df_final.astype(
    {
        'id': 'int32',
        'descricao_produto': 'string',
        'categoria': 'string',
        'id_fornecedor': 'int32',
        'valor': 'float32'
    }
)

df_format.dtypes

id                     int32
descricao_produto     string
categoria             string
id_fornecedor          int32
valor                float32
dtype: object

In [12]:
conn.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
        id BIGINT,
        descricao_produto TEXT,
        categoria TEXT,
        id_fornecedor BIGINT,
        valor FLOAT
        )
    """)

In [13]:
conn.execute("INSERT INTO produtos SELECT * FROM df_format")

In [14]:
tab_prod = conn.execute("SELECT * FROM produtos").fetchdf()
tab_prod.head(10)

,id,descricao_produto,categoria,id_fornecedor,valor
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10003,PREGO,BT10,100,60.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [15]:
conn.close()